# ATML Lab 6 Solutions
This notebook implements the practical tasks from the assignment using Encoder-Decoder architecture.
Tasks:
1. Design, implement and train an English to Hindi Machine Translation System.
2. Evaluate and analyse the performance.
4. Build any other real world application (Summarization) using Encoder-Decoder Architecture.
*Note: Step 3 was explicitly skipped per user instructions.*

In [ ]:
# Setup & Installs
%pip install -q torch torchtext --index-url https://download.pytorch.org/whl/cu121
%pip install -q pandas datasets transformers gradio

import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import pandas as pd
import os
from datasets import load_dataset
import math
import time

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Note: you may need to restart the kernel to use updated packages.


ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.
    torch from https://download.pytorch.org/whl/cu121/torch-2.2.0%2Bcu121-cp311-cp311-win_amd64.whl#sha256=d79324159c622243429ec214a86b8613c1d7d46fc4821374d324800f1df6ade1:
        Expected sha256 d79324159c622243429ec214a86b8613c1d7d46fc4821374d324800f1df6ade1
             Got        313240859a12e2f40a20e08ac308390d8ab90317fd48466ef4cf54e06cf4e760



Note: you may need to restart the kernel to use updated packages.
Using device: cuda


In [ ]:
# Task 1 & 2: Dataset Loading and Preprocessing
print("Loading dataset...")
try:
    if os.path.exists('Hindi_English_Truncated_Corpus.csv'):
        # Use kaggle dataset if available locally
        df = pd.read_csv('Hindi_English_Truncated_Corpus.csv')
        print("Loaded local Kaggle CSV.")
        from datasets import Dataset, DatasetDict
        dataset_dict = DatasetDict({
            'train': Dataset.from_pandas(df.iloc[:20000]),
            'validation': Dataset.from_pandas(df.iloc[20000:21000]),
            'test': Dataset.from_pandas(df.iloc[21000:22000])
        })
        dataset = dataset_dict
    else:
        # Fallback to standard open dataset
        print("CSV not found locally, defaulting to Hugging Face dataset...")
        dataset = load_dataset("cfilt/iitb-english-hindi")
        print("Loaded Hugging Face cfilt/iitb-english-hindi.")
except Exception as e:
    print(e)

Loading dataset...
CSV not found locally, defaulting to Hugging Face dataset...
Loaded Hugging Face cfilt/iitb-english-hindi.


In [ ]:
# Building vocabularies
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

en_tokenizer = get_tokenizer('basic_english')
hi_tokenizer = lambda x: x.split()

def yield_tokens(data_iter, lang):
    for example in data_iter:
        if 'translation' in example:
            yield en_tokenizer(example['translation']['en']) if lang == 'en' else hi_tokenizer(example['translation']['hi'])
        else: # For custom DataFrame parsing
            yield en_tokenizer(str(example['english_sentence'])) if lang == 'en' else hi_tokenizer(str(example['hindi_sentence']))

# Use subset for fast vocab building

en_vocab = build_vocab_from_iterator(yield_tokens(dataset['train'].select(range(min(10000, len(dataset['train'])))), 'en'), specials=['<unk>', '<pad>', '<sos>', '<eos>'])
en_vocab.set_default_index(en_vocab['<unk>'])

hi_vocab = build_vocab_from_iterator(yield_tokens(dataset['train'].select(range(min(10000, len(dataset['train'])))), 'hi'), specials=['<unk>', '<pad>', '<sos>', '<eos>'])
hi_vocab.set_default_index(hi_vocab['<unk>'])
print(f"English Vocab: {len(en_vocab)} | Hindi Vocab: {len(hi_vocab)}")

ModuleNotFoundError: No module named 'torchtext'

In [ ]:
# Dataloaders
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

BATCH_SIZE = 64
PAD_IDX = en_vocab['<pad>']
SOS_IDX = en_vocab['<sos>']
EOS_IDX = en_vocab['<eos>']

def data_process(data_iter):
    data = []
    for example in data_iter:
        en_text = example['translation']['en'] if 'translation' in example else str(example['english_sentence'])
        hi_text = example['translation']['hi'] if 'translation' in example else str(example['hindi_sentence'])
        
        en_tensor = torch.tensor([SOS_IDX] + [en_vocab[token] for token in en_tokenizer(en_text)] + [EOS_IDX], dtype=torch.long)
        hi_tensor = torch.tensor([SOS_IDX] + [hi_vocab[token] for token in hi_tokenizer(hi_text)] + [EOS_IDX], dtype=torch.long)
        data.append((en_tensor, hi_tensor))
    return data

train_data = data_process(dataset['train'].select(range(min(10000, len(dataset['train'])))))
valid_data = data_process(dataset['validation'])
test_data = data_process(dataset['test'])

def generate_batch(data_batch):
    en_batch, hi_batch = [], []
    for (en_item, hi_item) in data_batch:
        en_batch.append(en_item)
        hi_batch.append(hi_item)
    en_batch = pad_sequence(en_batch, padding_value=PAD_IDX)
    hi_batch = pad_sequence(hi_batch, padding_value=PAD_IDX)
    return en_batch, hi_batch

train_iterator = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=generate_batch)
valid_iterator = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=generate_batch)
test_iterator = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=generate_batch)

In [ ]:
# Encoder-Decoder Definition
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[0, :]
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1
        return outputs

INPUT_DIM = len(en_vocab)
OUTPUT_DIM = len(hi_vocab)
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

In [ ]:
# Training Loop (Task 1)
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        # Calculate gradients and clip them to avoid exploding gradients
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        # Update the model parameters
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0)
            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)
            loss = criterion(output, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)

N_EPOCHS = 10
CLIP = 1
best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_iterator, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_iterator, criterion)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'en-hi-model.pt')
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val. Loss: {valid_loss:.3f}')

In [ ]:
# Task 2: Evaluate and Analyse the Model
model.load_state_dict(torch.load('en-hi-model.pt'))
test_loss = evaluate(model, test_iterator, criterion)
print(f'Test Loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):7.3f}')

def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()
    tokens = ['<sos>'] + en_tokenizer(sentence) + ['<eos>']
    src_indexes = [src_vocab[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)
    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)
    trg_indexes = [trg_vocab['<sos>']]
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)
        if pred_token == trg_vocab['<eos>']:
            break
    itos = trg_vocab.get_itos()
    trg_tokens = [itos[i] for i in trg_indexes]
    return trg_tokens[1:]

src_txt = "my name is" # example text
pred = translate_sentence(src_txt, en_vocab, hi_vocab, model, device)
print(f'Source: {src_txt}')
print(f'Predicted Translation: {" ".join(pred[:-1])}')

In [ ]:
# Task 4: Another Real-World Application using Encoder-Decoder (Summarization)
from transformers import pipeline

print("========== Task 4: Summarization Application ==========")
summarizer = pipeline("summarization", model="t5-small", device=0 if torch.cuda.is_available() else -1)

ARTICLE = """
Machine translation is a sub-field of computational linguistics that investigates the use of software to translate text or speech from one language to another. 
On a basic level, MT performs mechanical substitution of words in one language for words in another, but that alone rarely produces a good translation 
because recognition of whole phrases and their closest counterparts in the target language is needed. 
The advent of deep learning has revolutionized machine translation, particularly through sequence-to-sequence models and the Transformer architecture.
"""

summary = summarizer(ARTICLE, max_length=50, min_length=20, do_sample=False)
print("Original Text:\n", ARTICLE)
print("Generated Summary:\n", summary[0]['summary_text'])

In [ ]:
# BONUS: Interactive Web Interface (Gradio)
import gradio as gr

print("========== Launching Web Interface ==========")

def web_translate(text):
    try:
        pred = translate_sentence(text, en_vocab, hi_vocab, model, device)
        return ' '.join(pred[:-1])
    except Exception as e:
        return str(e)

def web_summarize(text):
    try:
        out = summarizer(text, max_length=50, min_length=10, do_sample=False)
        return out[0]['summary_text']
    except Exception as e:
        return str(e)

with gr.Blocks() as demo:
    gr.Markdown("# ATML Lab 6: Encoder-Decoder Models")
    
    with gr.Tab("English to Hindi Translation (Custom model)"):
        trans_input = gr.Textbox(lines=2, placeholder="Enter English text...")
        trans_output = gr.Textbox(lines=2, label="Hindi Translation")
        trans_btn = gr.Button("Translate")
        trans_btn.click(fn=web_translate, inputs=trans_input, outputs=trans_output)
        
    with gr.Tab("Summarization (T5 Architecture)"):
        sum_input = gr.Textbox(lines=5, placeholder="Enter long English text...")
        sum_output = gr.Textbox(lines=3, label="Summary")
        sum_btn = gr.Button("Summarize")
        sum_btn.click(fn=web_summarize, inputs=sum_input, outputs=sum_output)

demo.launch(share=True, quiet=True)


In [ ]:
#to only extract numbers without spaces from a string
import string
str="""sjcnsanpewjmq2309402-1-04234k
i 34j2r,wm wekm#@($)  INSFENW V EWKMREPOM&@
*#&!(*)V,Wvrfn ewonimvwpwmIPVMERQPNMRo2893
04893"""
allowed = set(string.digits + string.punctuation+"\n")
numbers = ''.join(c for c in str if c in allowed)
print(numbers)
#printing the string after removing the numbers and punctuation
print(''.join(c for c in str if c not in allowed and not c.isspace()))

2309402-1-04234
342,#@($)&@
*#&!(*),2893
04893
sjcnsanpewjmqkijrwmwekmINSFENWVEWKMREPOMVWvrfnewonimvwpwmIPVMERQPNMRo


In [ ]:
import re
data = """
TRADE#ID: 98342 | Name: Yash Kothari | Age=19 | Email: yashk@gmail.com | Price: $1200.50 | Date: 2026-03-21
ID-88321 Name:Riya Sharma Age:20 Email:riya_sharma99@mail.com Price=950 Date=21/03/2026
User Rahul(21) | Mail: rahul21@finance.org | Trade Value: 1,200 USD | Ref: TRX9921
#ID:77123;Name=Arjun Verma;Age=22;Email=arjun.v@email.net;Price:$2,500.75;Date:2026.03.20
Client: Sneha | Age : 23 | email sneha23@mail | Amount: 3000 INR | Date missing
ID: 66521 Name: Vikram Singh Age: Twenty Four Email: vikram@mail.com Price: $1800 Date: 2026-03-19
"""
#1. Extracting all email addresses from the data
print("Extracted emails: ")
emails=re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\s+', data)
print(emails)
#2 extracting all numbers from the data
s=''.join(c for c in data if c.isdigit())
print(s)
#3. Extracting all names from the data
data1=data.split("Name")
for name in data1[1:]:
    name_part=name.split("Age")[0].strip()
    names=''.join(c for c in name_part if c.isalpha() or c.isspace())
    names=names.strip()
    print(names)
#
data2=re.findall(r'\d+.\d+', data)
print(data2)
a=[]
data3=re.findall(r'Price:\s*\$?\d+(?:\.\d+,+\d+)?', data)
print(data3)

Extracted emails: 
['yashk@gmail.com ', 'riya_sharma99@mail.com ', 'rahul21@finance.org ', 'vikram@mail.com ']
983421912005020260321883212099950210320262121120099217712322250075202603202323300066521180020260319
Yash Kothari
Riya Sharma
Arjun Verma
Vikram Singh
['98342', '1200.50', '2026-03', '88321', '950', '21/03', '2026', '1,200', '9921', '77123', '2,500', '2026.03', '3000', '66521', '1800', '2026-03']
['Price: $1200', 'Price:$2', 'Price: $1800']
